**Semantic search project: Assistant**

---


it mainly uses tranformer encodings to encode chunks of texts taken from pdfs and according to a query it gives the closest chunks to it in terms of embeddings. It manily applies the transformers and semantic search consepts

In [ ]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf
!pip install streamlit

In [ ]:
!pip install -q google-genai

In [ ]:
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
import faiss
import numpy as np

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from pathlib import Path
from pypdf import PdfReader

pdf_folder = Path(".")

documents = []

for pdf in pdf_folder.glob("*.pdf"):

    reader = PdfReader(pdf)

    text = ""

    for page_number, page in enumerate(reader.pages):

        if page.extract_text():

            text += page.extract_text()

    documents.append({
        "source": pdf.name,
        "page": page_number + 1,
        "text": text
    })

In [ ]:
chunks = []

for doc in documents:

    words = doc["text"].split()

    chunk_size = 200

    overlap = 50

    for i in range(
        0,
        len(words),
        chunk_size - overlap
    ):

        chunk = " ".join(
            words[i:i+chunk_size]
        )

        chunks.append(
            {
                "text": chunk,
                "source": doc["source"],
                "page": doc["page"],
                "chunk_id": len(chunks)
            }
        )

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True
)

In [ ]:
print(type(embeddings))
print(embeddings.shape)
print(len(chunks))

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings).astype("float32"))

In [ ]:
def retrieve(query, k=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    D, I = index.search(
        query_embedding.astype("float32"),
        k
    )

    retrieved = []

    for idx in I[0]:
        retrieved.append(chunks[idx])

    return retrieved

In [ ]:
query="what is the design of the antenna?"
results = retrieve(query)


In [ ]:

for result in results:
    print(result["source"])
    print(result["page"])
    print(result["text"])

In [ ]:
context = ""

for chunk in results:

    context += f"""
Source: {chunk['source']}
Page: {chunk['page']}

{chunk['text']}

------------------
"""